# 09 Sarvam-Translate — Colab Setup & Verification

> **Before running:** `Runtime > Change runtime type > T4 GPU`

Uses `sarvamai/sarvam-translate` locally on Colab GPU — no API key, no credits.

**Steps:**
1. Upload `fakehealth_healthfact_binary_clean.csv` to your Google Drive
2. Set `DATASET_PATH` in Cell 4 to match where you put it
3. Run all cells top to bottom


## 1. Check GPU


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name :', torch.cuda.get_device_name(0))
    print('VRAM     :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU and re-run.')


## 2. Install Packages


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'transformers','accelerate','sentencepiece','-q'])
print('Packages ready.')


## 3. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


## 4. Paths

> **Edit these two paths** to match where your files are in Google Drive.


In [ ]:
from pathlib import Path

# ── EDIT THESE ────────────────────────────────────────────────────────
DATASET_PATH = Path('/content/drive/MyDrive/Modar Riba/Dataset/dataset/processed/fakehealth_healthfact_binary_clean.csv')
TRANS_ROOT   = Path('/content/drive/MyDrive/Modar Riba/Translation')
# ──────────────────────────────────────────────────────────────────────

TRANS_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset path :', DATASET_PATH)
print('Output root  :', TRANS_ROOT)
print('Dataset exists:', DATASET_PATH.exists())


## 5. Load Model


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'sarvamai/sarvam-translate'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_NAME} on {DEVICE} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)
model.eval()
print('Model ready.')


## 6. Translation Helpers


In [ ]:
def split_sentences(text: str, max_sent: int = 5) -> list:
    parts = [s.strip() for s in text.replace('. ', '.|||').split('|||') if s.strip()]
    return [' '.join(parts[i:i+max_sent]) for i in range(0,len(parts),max_sent)] or [text]


def translate_chunk(chunk: str, tgt_lang: str) -> str:
    messages = [
        {'role':'system','content':f'Translate the text below to {tgt_lang}.'},
        {'role':'user',  'content': chunk},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False, num_return_sequences=1,
        )
    out_ids = gen_ids[0][len(inputs.input_ids[0]):].tolist()
    return tokenizer.decode(out_ids, skip_special_tokens=True).strip()


def translate_text(text: str, tgt_lang: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    return ' '.join(translate_chunk(c, tgt_lang) for c in split_sentences(text))


print('Helpers ready.')


## 7. Language Map


In [ ]:
LANGUAGE_MAP = {
    'assamese' : 'Assamese',
    'hindi'    : 'Hindi',
    'manipuri' : 'Manipuri',
    'bodo'     : 'Bodo',
}
print('Language map ready.')


## 8. Test All 4 Languages


In [ ]:
import time
TEST = 'Regular physical activity reduces the risk of heart disease and diabetes.'
print(f'Source: {TEST}\n' + '-'*65)

for lang_key, lang_str in LANGUAGE_MAP.items():
    t0  = time.time()
    out = translate_chunk(TEST, lang_str)
    print(f'[{lang_key:<12}] {round(time.time()-t0,2)}s')
    print(f'  {out}\n')


## 9. Test with a Real Dataset Row


In [ ]:
import pandas as pd
df     = pd.read_csv(DATASET_PATH)
sample = df.iloc[0]
print('Dataset shape:', df.shape)
print('\nTranslating first row to Hindi...')
t0       = time.time()
title_hi = translate_chunk(str(sample['title']), 'Hindi')
text_hi  = translate_text(str(sample['text']),   'Hindi')
print(f'\nOriginal title : {sample["title"]}')
print(f'Hindi title    : {title_hi}')
print(f'\nHindi text     : {text_hi[:300]}...')
print(f'Time           : {round(time.time()-t0,1)}s')
